In [35]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import pandas as pd
import time
import openpyxl

In [36]:
# Criar navegador no Chrome

navegador = webdriver.Chrome()

In [37]:
# lendo a base de dados

df = pd.read_excel(r'4_Base de dados de produtos\Produtos.xlsx')

display(df)

,nome,autor,categoria,preco,link
0,Frankenstein,Mary Shelley,Classics,NaN,NaN
1,Romeo and Juliet,Shakespeare,Classics,NaN,NaN
2,The Great Gatsby,Fitzgerald,Classics,NaN,NaN
3,Algorithms to Live By,Brian Christian,Nonfiction,NaN,NaN
4,Sapiens,Yuval Harari,History,NaN,NaN
5,Smarter Faster Better,Charles Duhigg,Nonfiction,NaN,NaN


### Função de pesquisa no site Gutenberg

In [38]:

def pesquisa_gutenberg(nome, autor, navegador):
    navegador.get("https://gutenberg.org")
    
    try:
        busca = navegador.find_element("class name", "search-input") #encontrar o elemento na página (botão de busca) 
        busca.send_keys(nome) # insere o item de pesquisa
        busca.send_keys(Keys.ENTER)
                
        lista_resultados = navegador.find_elements("class name", "booklink")
        for resultado in lista_resultados: 
            titulo = resultado.text
            if nome.lower() in titulo.lower():
                # Verifica se as palavras do autor estão no texto
                if all(palavra.lower() in titulo.lower() for palavra in autor.split()):
                    link = resultado.find_element("class name","link").get_attribute("href")
                    preco = '0'
                    return link, preco
    except Exception as e:
        print(f"Erro no Gutenberg: {e}")
        
    return None, None


def pesquisa_books(nome, categoria, navegador):
    navegador.get('https://books.toscrape.com/')
    
    # Seleção de categoria
    try:
        navegador.find_element('class name', 'nav-list').find_element('link text', categoria).click()
    except NoSuchElementException:
        print(f'Categoria "{categoria}" não encontrada.')

    while True:
        # IMPORTANTE: A lista deve ser atualizada TODA VEZ que a página mudar
        lista_resultados = navegador.find_elements("class name", "product_pod")
        
        for resultado in lista_resultados: 
            try:
                tag_a = resultado.find_element('tag name', 'h3').find_element('tag name', 'a')
                titulo = tag_a.get_attribute('title')
                
                if nome.lower() in titulo.lower():
                    link = tag_a.get_attribute("href")
                    preco = resultado.find_element("class name", "price_color").text
                    return link, preco
            except StaleElementReferenceException:
                # Se a página mudar enquanto lê, interrompe e tenta a nova página
                break 

        # Tenta ir para a próxima página
        try:
            botao_next = navegador.find_element("link text", "next")
            botao_next.click()
            time.sleep(2) # Pequena pausa para carregar o DOM
        except NoSuchElementException:
            break # Fim das páginas
                
    return None, None




In [39]:
# Loop Principal
for linha in df.index:
    nome = df.loc[linha, 'nome']
    autor = df.loc[linha, 'autor']
    categoria = df.loc[linha, 'categoria']

    link1, preco1 = pesquisa_gutenberg(nome, autor, navegador)
    
    if link1:
        print(f"✅ Encontrado no Gutenberg: {nome} | {link1} | {preco1}")
    else: 
        link2, preco2 = pesquisa_books(nome, categoria, navegador)
        if link2:
            print(f"✅ Encontrado no Bookscrape: {nome} | {link2} | {preco2}")
        else:
            print(f"❌ '{nome}' não encontrado.")

# navegador.quit() # Opcional: fechar ao terminar


Categoria "Classics" não encontrada.
❌ 'Frankenstein' não encontrado.
✅ Encontrado no Gutenberg: Romeo and Juliet | https://gutenberg.org/ebooks/1513 | 0
✅ Encontrado no Gutenberg: The Great Gatsby | https://gutenberg.org/ebooks/64317 | 0
✅ Encontrado no Bookscrape: Algorithms to Live By | https://books.toscrape.com/catalogue/algorithms-to-live-by-the-computer-science-of-human-decisions_880/index.html | £30.81
✅ Encontrado no Bookscrape: Sapiens | https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html | £54.23
✅ Encontrado no Bookscrape: Smarter Faster Better | https://books.toscrape.com/catalogue/smarter-faster-better-the-secrets-of-being-productive-in-life-and-business_543/index.html | £38.89


In [31]:
import pandas as pd
from selenium.common.exceptions import WebDriverException

# ... (suas funções pesquisa_gutenberg e pesquisa_books continuam iguais) ...

try:
    for linha in df.index:
        nome = df.loc[linha, 'nome']
        autor = df.loc[linha, 'autor']
        categoria = df.loc[linha, 'categoria']

        try:
            # 1. Tenta a primeira fonte
            link, preco = pesquisa_gutenberg(nome, autor, navegador)
            
            # 2. Se não encontrar, tenta a segunda
            if not link:
                link, preco = pesquisa_books(nome, categoria, navegador)
            
            # 3. Atualiza o DataFrame
            if link:
                df.at[linha, 'link'] = link
                df.at[linha, 'preco'] = preco
                print(f"✅ Atualizado: {nome} -> {preco}")
            else:
                print(f"❌ Não encontrado: {nome}")

        except WebDriverException:
            print(f"⚠️ Erro de conexão no item '{nome}'. O navegador pode ter sido fechado.")
            break # Sai do loop para salvar o que já foi processado
        except Exception as e:
            print(f"🔍 Erro inesperado no item {nome}: {e}")
            continue # Pula para o próximo livro

finally:
    # 4. SALVAR O ARQUIVO (Sempre executa, mesmo se houver erro no loop)
    caminho_salvar = r'4_Base de dados de produtos\Produtos_Atualizados.xlsx'
    df.to_excel(caminho_salvar, index=False)
    print(f"\n--- Processo finalizado! Arquivo salvo em: {caminho_salvar} ---")

❌ Não encontrado: Frankenstein
🔍 Erro inesperado no item Romeo and Juliet: Invalid value 'https://gutenberg.org/ebooks/1513' for dtype 'float64'
🔍 Erro inesperado no item The Great Gatsby: Invalid value 'https://gutenberg.org/ebooks/64317' for dtype 'float64'
🔍 Erro inesperado no item Algorithms to Live By: Invalid value 'https://books.toscrape.com/catalogue/algorithms-to-live-by-the-computer-science-of-human-decisions_880/index.html' for dtype 'float64'
🔍 Erro inesperado no item Sapiens: Invalid value 'https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html' for dtype 'float64'
🔍 Erro inesperado no item Smarter Faster Better: Invalid value 'https://books.toscrape.com/catalogue/smarter-faster-better-the-secrets-of-being-productive-in-life-and-business_543/index.html' for dtype 'float64'

--- Processo finalizado! Arquivo salvo em: 4_Base de dados de produtos\Produtos_Atualizados.xlsx ---


In [32]:
# --- SCRIPT PRINCIPAL ---

# 1. Configuração Inicial
navegador = webdriver.Chrome()
caminho_arquivo = r'4_Base de dados de produtos\Produtos.xlsx'
df = pd.read_excel(caminho_arquivo)

# Garante que as colunas existam
if 'link' not in df.columns: df['link'] = ""
if 'preco' not in df.columns: df['preco'] = ""

try:
    for linha in df.index:
        nome = df.loc[linha, 'nome']
        autor = df.loc[linha, 'autor']
        categoria = df.loc[linha, 'categoria']

        try:
            # 2. Executa as buscas
            link, preco = pesquisa_gutenberg(nome, autor, navegador)
            
            if not link:
                link, preco = pesquisa_books(nome, categoria, navegador)
            
            # 3. Atualiza o DataFrame (Tratando tudo como String)
            if link:
                df.at[linha, 'link'] = str(link)
                # Limpa o caractere 'Â' que costuma aparecer em erros de encoding do símbolo £
                preco_string = str(preco).replace('Â', '').strip()
                df.at[linha, 'preco'] = preco_string
                print(f"✅ Sucesso: {nome} -> {preco_string}")
            else:
                print(f"❌ Não encontrado: {nome}")

        except WebDriverException:
            print(f"⚠️ Conexão perdida com o navegador ao buscar '{nome}'.")
            break 
        except Exception as e:
            print(f"🔍 Erro inesperado no item '{nome}': {e}")
            continue

finally:
    # 4. Salva o progresso independente de erros
    caminho_final = r'4_Base de dados de produtos\Produtos_Atualizados.xlsx'
    df.to_excel(caminho_final, index=False)
    navegador.quit()
    print(f"\n--- Processo finalizado! Planilha salva em: {caminho_final} ---")

🔍 Erro inesperado no item 'Frankenstein': Invalid value 'https://gutenberg.org/ebooks/84' for dtype 'float64'
🔍 Erro inesperado no item 'Romeo and Juliet': Invalid value 'https://gutenberg.org/ebooks/1513' for dtype 'float64'
🔍 Erro inesperado no item 'The Great Gatsby': Invalid value 'https://gutenberg.org/ebooks/64317' for dtype 'float64'
🔍 Erro inesperado no item 'Algorithms to Live By': Invalid value 'https://books.toscrape.com/catalogue/algorithms-to-live-by-the-computer-science-of-human-decisions_880/index.html' for dtype 'float64'
🔍 Erro inesperado no item 'Sapiens': Invalid value 'https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html' for dtype 'float64'
🔍 Erro inesperado no item 'Smarter Faster Better': Invalid value 'https://books.toscrape.com/catalogue/smarter-faster-better-the-secrets-of-being-productive-in-life-and-business_543/index.html' for dtype 'float64'

--- Processo finalizado! Planilha salva em: 4_Base de dados de produtos\Produto